### Imports

In [13]:
from testgen.utils import *
import pandas as pd
import os
from pathlib import Path
from dotenv import load_dotenv

In [14]:
base_path = Path().cwd()
load_dotenv(override=True)

True

In [15]:
# read data and rename columns
df = pd.read_excel(base_path / "data/sensor_requirements.xlsx")
df_examples = pd.read_excel(base_path / "data/sensor_examples.xlsx")

# Find Examples

In [16]:
examples_txt, examples = get_example_txt(df_examples, 1)

print("\n".join(examples_txt.split("\n")[-10:]))

Requirement: Drift control systems must adaptively use yaw rate feedback to optimize vehicle control during high-performance driving scenarios
Target Sensor/s: [yaw_rate]

Requirement: The power steering system must adapt the torque levels in response to detected road surface conditions (e.g., ice, water, gravel)
Target Sensor/s: [steering_torque]

Requirement: The control system must detect and mitigate power oversteer to maintain vehicle control by adjusting the wheel steering angle and acceleration pedal inputs
Target Sensor/s: [acceleration_pedal, wheel_steering_angle]




# Response Format

In [17]:
from testgen.prompts.Sensors import Sensors
from typing import List
from pydantic import BaseModel, Field, create_model

Sensors_t = Sensors.split("\n")


def clean_sensors_fn(x):
    x = x.split(":")
    x[0] = x[0][: x[0].find("(")]
    return x


Sensors_t = list(map(clean_sensors_fn, Sensors_t))

sensor_attrs = {}

for sensor in Sensors_t:
    sensor_attrs[sensor[0].strip().lower().replace(" ", "_")] = (
        int,
        Field(description=sensor[1].strip()),
    )


TargetSensor = create_model("TargetSensor", **sensor_attrs)


class TargetSensorList(BaseModel):
    req_id: int = Field(description="Reauirement's ID")
    text: str = Field(description="Reauirement's text")
    target_sensor: TargetSensor = Field(
        description="List of sensors where 1 is the targeted sensor and 0 is not"
    )


class RequirementList(BaseModel):
    requirements: List[TargetSensorList] = Field(
        description="List of requirements and their TargetSensorList"
    )

In [18]:
# print(json.dumps(TargetSensor.model_json_schema(), indent=2))

In [19]:
# print(json.dumps(TargetSensorList.model_json_schema(), indent=2))

# LLM

In [20]:
from testgen.prompts import SystemPrompt
from testgen.prompts import Sensors
from testgen.prompts import UserPromptBulk

In [29]:
llm_models = {
    # "azure": ["gpt-4o-mini", "gpt-4o"],
    "novita": [
        "qwen/qwen2.5-7b-instruct",
        "meta-llama/llama-3-70b-instruct",
    ],
}

endpoint_attrs = {
    "azure": {
        "api_key": os.getenv("AZURE_OPENAI_API_KEY"),
        "api_version": os.getenv("AZURE_API_VERSION"),
        "base_url": os.getenv("AZURE_OPENAI_ENDPOINT"),
    },
    "novita": {
        "api_key": os.getenv("NOVITA_API_KEY"),
        "base_url": os.getenv("NOVITA_ENDPOINT"),
    },
}

### Get Requirements for multi prediction

In [30]:
batches = get_batches(df, "random", 3)
batches, req_texts = requirement_text_bulk(batches, df.columns)

Number of Batches: 32
Number of Instances left: 1


In [31]:
print(req_texts[0])

<ID> 21 <Text> The system must periodically verify the calibration of each wheel speed sensor and recalibrate if necessary.
<ID> 77 <Text> Vehicle control systems must respond to yaw rate sensing  by adjusting braking, throttle, and steering as necessary to maintain intended vehicle path and stability.
<ID> 51 <Text> The steering system must remain operable within safe limits in the event of a torque sensor failure.



In [32]:
SAMPLE_TYPE = "random"

for N_EXAMPLES in [8]:

    examples_txt, examples = get_example_txt(df_examples, N_EXAMPLES)

    for batch_size in [2, 3, 5]:

        batches = get_batches(df, SAMPLE_TYPE, batch_size)
        batches, req_texts = requirement_text_bulk(batches, df.columns)

        for endpoint_name in llm_models.keys():

            for model_name in llm_models[endpoint_name]:

                print(
                    f"Running {model_name} on {endpoint_name} examples {N_EXAMPLES} batch_size {batch_size}..."
                )

                client = llm_client(endpoint_name, **endpoint_attrs[endpoint_name])

                results = invoke_bulk_sensor(
                    endpoint_name,
                    model_name,
                    client,
                    batches,
                    req_texts,
                    examples_txt,
                    SystemPrompt,
                    Sensors,
                    UserPromptBulk,
                    # limit_batch_size=1,
                    response_format=RequirementList,
                )

                (
                    number_of_requests,
                    accuracy,
                    avg_time_per_req,
                    avg_token_per_req,
                    avg_completion_token_per_req,
                    total_tokens,
                    total_completion_tokens,
                    total_time,
                ) = calc_stats(results, number_of_requests=len(batches))

                results_file = save_responses(
                    base_path,
                    "sensor_bulk",
                    model_name=model_name.split("/")[-1],
                    n_examples=N_EXAMPLES,
                    batch_size=batch_size,
                    examples=examples,
                    accuracy=accuracy,
                    number_of_requests=number_of_requests,
                    total_tokens=total_tokens,
                    total_completion_tokens=total_completion_tokens,
                    avg_token_per_req=avg_token_per_req,
                    avg_completion_token_per_req=avg_completion_token_per_req,
                    avg_time_per_req=avg_time_per_req,
                    results=results,
                )

Number of Batches: 48
Number of Instances left: 1
Running qwen/qwen2.5-7b-instruct on novita examples 8 batch_size 2...


  0%|          | 0/48 [00:00<?, ?it/s]

Running meta-llama/llama-3-70b-instruct on novita examples 8 batch_size 2...


  0%|          | 0/48 [00:00<?, ?it/s]

Number of Batches: 32
Number of Instances left: 1
Running qwen/qwen2.5-7b-instruct on novita examples 8 batch_size 3...


...

Number of Batches: 19
Number of Instances left: 2
Running qwen/qwen2.5-7b-instruct on novita examples 8 batch_size 5...


  0%|          | 0/19 [00:00<?, ?it/s]

Running meta-llama/llama-3-70b-instruct on novita examples 8 batch_size 5...


  0%|          | 0/19 [00:00<?, ?it/s]

KeyboardInterrupt: 